# V2.1-C: Avaliação confirmatória de stress (passo 7)

Este notebook é um auxiliar de relatório, somente texto, como os
notebooks 11 e 12: carrega o `v2_stress_protocol.json` congelado, o
resultado agregado selado e o pequeno manifesto publicado, e os
imprime juntos como um único relatório em texto simples. Não há
modelagem, não há gráficos e não há dataframes aqui -- apenas leitura
e impressão do que a execução selada publicou.

O passo 7, regido por
`docs/ADR-014-v2-stress-confirmatory-evaluation.md`, é a abertura
única da partição `stress`, de 2025-07-01 a 2025-12-31. A ADR-014
promove `stress` de partição diagnóstica a confirmatória, porque
`test` -- a partição desenhada para confirmar -- já foi consumida pelo
S8 e é proibida ao V2 pela ADR-010.

Uma única passagem sobre o dado selado pontua dois braços:

- `v2_combined`: o braço primário, o estágio A sobrescrevendo o
  fallback S7 congelado sempre que sua margem alcança o limiar
  calibrado;
- `s7_fallback_alone`: o controle S7 congelado, sobre exatamente as
  mesmas linhas.

Os arquivos de entrada, quando presentes, são:

- `config/v2_stress_protocol.json`: a pré-registração congelada,
  sempre presente, mesmo antes da execução selada acontecer;
- `temp/v2/v2_stress_results.json`: o resultado agregado completo,
  escrito somente após a abertura selada de execução única;
- `config/v2_stress_results.json`: o pequeno manifesto publicado.

Este notebook não lê o token de destravamento, não o guarda e não pode
disparar a execução selada por si só. Ele apenas exibe evidência
agregada já publicada.

## Por que o contraste pareado é a comparação primária

`stress` 2025-H2 tem composição de classes diferente das janelas em
que o V2 foi desenvolvido e calibrado: em grupos inéditos,
`money_services` cai 62 por cento, `student_loan` cai 44 por cento,
`credit_reporting` cai 22 por cento e a classe crítica cai 16 por
cento em relação a `test`. Como macro-F1 é a média não ponderada de
nove classes, um número absoluto do V2 em `stress` não é comparável ao
número do S8 em `test`, mesmo para o modelo idêntico: o mesmo S7
congelado, sem nenhuma alteração, já marcou F1 crítico 0,339665 em
`validation` e 0,257843 em `test`, uma diferença de 0,081822
atribuível somente à janela.

O contraste pareado é imune a essa deriva porque os dois braços são
pontuados sobre as mesmas linhas, na mesma passagem. Quatro gates
decidem o desfecho, avaliados somente na visão científica: os três
pisos absolutos herdados do S8 sem alteração (macro-F1 de pelo menos
0,69, F1 crítico de pelo menos 0,2715, precisão crítica de pelo menos
0,20), e um gate pareado e estrito -- o F1 crítico do V2 combinado
precisa ser estritamente maior que o do S7 sozinho nas mesmas linhas.
Com 4 de 4, o status é `CONFIRMED`; caso contrário, `NOT_CONFIRMED`. O
ganho pareado de desenvolvimento, 0,047234, é uma expectativa
pré-registrada, não um gate.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.v2_import import (
    load_stress_payload,
    render_stress_import_report,
)

In [2]:
STRESS_PROTOCOL_PATH = PROJECT_ROOT / 'config' / 'v2_stress_protocol.json'

protocol_payload = load_stress_payload(STRESS_PROTOCOL_PATH)

In [3]:
STRESS_RESULT_PATH = PROJECT_ROOT / 'temp' / 'v2' / 'v2_stress_results.json'
STRESS_MANIFEST_PATH = PROJECT_ROOT / 'config' / 'v2_stress_results.json'

result_payload = load_stress_payload(STRESS_RESULT_PATH)
manifest_payload = load_stress_payload(STRESS_MANIFEST_PATH)

In [4]:
print(render_stress_import_report(
    result_payload,
    manifest_payload,
    protocol_payload,
))

V2.1-C STRESS CONFIRMATORY EVALUATION

VERDICT
  stage: V2.1-C   adr: ADR-014
  stress_scope: 2025-07-01 to 2025-12-31
  status: NOT_CONFIRMED
  confirmed: False
  deploy: False
  gates_passed: 3 / 4

  *** NOT_CONFIRMED: at least one pre-registered gate failed on the sealed stress partition. ***
  A CONFIRMED verdict never authorizes deployment; deploy is always false.

GATES (SCIENTIFIC VIEW, 4 REQUIRED)
     gate                     observed  limit     strict  verdict
  -  -----------------------  --------  --------  ------  -------
     macro_f1                 0.710748  0.690000  False   PASS   
  !  critical_f1              0.260404  0.271500  False   FAIL   
     critical_precision       0.426070  0.200000  False   PASS   
     paired_critical_f1_gain  0.006455  0.000000  True    PASS   
  ! marks a failed gate.

  *** AT LEAST ONE GATE FAILED. ***

THE PAIRED CONTRAST (PRIMARY, DRIFT-CONTROLLED)
  v2_combined critical_f1:       0.260404
  s7_fallback_alone critical_f1: 0.253949

## Lendo o veredito

VERDICT traz o status, a flag confirmed e o fato de `deploy` ser
sempre `false`: um veredito `CONFIRMED` nunca autoriza deployment,
apenas que o pacote congelado passou nos quatro gates pré-registrados
numa janela nunca vista. THE PAIRED CONTRAST é o bloco imune à deriva:
os mesmos dois braços, as mesmas linhas, uma única abertura do selo.
EXPECTATION CHECK é diagnóstico, não um gate, e apenas informa se o
ganho observado concorda em sinal com o ganho de desenvolvimento
pré-registrado.

Um veredito `NOT_CONFIRMED` encerra o ciclo do V2 com a medida
publicada, exatamente como o S8 encerrou o V1; não abre um V2.2 sobre
`stress`. `monitor` 2026 permanece selado e diagnóstico de qualquer
forma, e esta foi a última leitura independente disponível ao V2.